# Hybrid CVRP Solver — QAOA Cut-Point Selection

**Algorithm:**
1. **Classical** — encode locations, build Euclidean distance matrix, sort customers by polar angle from depot.
2. **QAOA** — find the optimal placement of G−1 cut points among the H−1 angular gaps, minimizing total route distance subject to exactly G−1 cuts.
3. **Classical** — optimal route ordering within each group via exhaustive permutation search.

**QUBO formulation:** binary variable $b_i = 1$ means a cut between angular neighbors $i$ and $i+1$.

$$C(b) = \sum_i w_i b_i + \lambda\left(\sum_i b_i - (G-1)\right)^2$$

where $w_i = d(c_i, \text{depot}) + d(\text{depot}, c_{i+1}) - d(c_i, c_{i+1})$ is the detour cost of routing through the depot instead of directly between angular neighbors. Via $b_i = (1-Z_i)/2$ this maps to a standard Ising Hamiltonian solved by QAOA.

In [ ]:
import math, os, time
import numpy as np
from itertools import permutations
from scipy.optimize import minimize

from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_aer import AerSimulator

service = QiskitRuntimeService(channel="ibm_quantum_platform", token="_m8-ZFH-nbDuptKu7e2vwyq6qlCNP9bErPAW2AUGFroJ")
hw_backend = service.least_busy(simulator=False, operational=True, min_num_qubits=12)
sim_backend = AerSimulator()

# Raised to 15 — ibm_fez has 156 qubits, all 4 instances (max 11 qubits) run on hardware
HW_QUBIT_THRESHOLD = 15

print(f"Hardware backend: {hw_backend.name}")
print(f"Circuits with >{HW_QUBIT_THRESHOLD} qubits will use local AerSimulator")

## Classical Utilities

In [42]:
def encode_locations(depot, customers):
    locations = {0: tuple(depot)}
    for i, c in enumerate(customers, start=1):
        locations[i] = tuple(c)
    return locations

def euclidean(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def build_distance_matrix(locations):
    ids = list(locations.keys())
    return {(i,j): euclidean(locations[i], locations[j]) for i in ids for j in ids if i != j}

def compute_num_groups(H, C, N):
    return min(math.ceil(H/C), N)

def angular_sort(customer_ids, locations):
    ox, oy = locations[0]
    return sorted(customer_ids, key=lambda n: math.atan2(locations[n][1]-oy, locations[n][0]-ox))

## QUBO: Cut-Point Weights

$w_i > 0$ means cutting there is expensive (the two customers are close together, so routing through the depot adds significant distance). QAOA will prefer to place cuts where $w_i$ is small.

In [43]:
def build_cut_weights(sorted_custs, dist_matrix):
    w = []
    for i in range(len(sorted_custs) - 1):
        ci, cj = sorted_custs[i], sorted_custs[i+1]
        w.append(dist_matrix[(ci, 0)] + dist_matrix[(0, cj)] - dist_matrix[(ci, cj)])
    return w

## Cost Hamiltonian

Via $b_i = (1-Z_i)/2$, the QUBO maps to the Ising Hamiltonian (dropping constants):

$$H_C = \sum_i \left(-\frac{w_i}{2} - \lambda A\right) Z_i + \frac{\lambda}{2}\sum_{i<j} Z_i Z_j$$

where $A = n/2 - (G-1)$ and $n$ is the number of binary variables (gaps).

Qiskit convention: qubit $i$ maps to string position $n-1-i$ (rightmost = qubit 0).

In [44]:
def build_cost_hamiltonian(weights, G):
    n = len(weights)
    lam = max(abs(w) for w in weights) * n * 2
    A = n/2 - (G-1)

    terms = []

    for i, w in enumerate(weights):
        coeff = -(w/2) - lam*A
        z = list('I'*n)
        z[n-1-i] = 'Z'
        terms.append((''.join(z), coeff))

    for i in range(n):
        for j in range(i+1, n):
            zz = list('I'*n)
            zz[n-1-i] = 'Z'
            zz[n-1-j] = 'Z'
            terms.append((''.join(zz), lam/2))

    return SparsePauliOp.from_list(terms)

## QAOA

Uses `QAOAAnsatz` (Qiskit) with COBYLA optimization. The expectation value is computed via exact statevector simulation, making this compatible with any qBraid-provided statevector backend.

In [ ]:
def run_qaoa(cost_op, reps=1, shots=2048):
    n_qubits = cost_op.num_qubits
    use_hardware = n_qubits <= HW_QUBIT_THRESHOLD
    active_backend = hw_backend if use_hardware else sim_backend
    backend_label = hw_backend.name if use_hardware else "aer_simulator (local)"

    print(f"  Running on: {backend_label}")

    ansatz = QAOAAnsatz(cost_op, reps=reps)
    params = ansatz.parameters
    x0 = np.random.uniform(0, np.pi, len(params))

    # Classical optimization loop via local statevector
    count = [0]
    def objective(x):
        count[0] += 1
        if count[0] % 25 == 0:
            print(f"    iteration {count[0]}")
        bound = ansatz.assign_parameters(dict(zip(params, x)))
        return Statevector(bound).expectation_value(cost_op).real

    t0 = time.time()
    opt = minimize(objective, x0, method='COBYLA',
                   options={'maxiter': 200, 'rhobeg': 0.5})
    elapsed = time.time() - t0

    # Transpile for the target backend
    bound_final = ansatz.assign_parameters(dict(zip(params, opt.x)))
    bound_final.measure_all()

    pm = generate_preset_pass_manager(optimization_level=2, backend=active_backend)
    transpiled = pm.run(bound_final)

    ops      = transpiled.count_ops()
    gates    = sum(ops.values())
    cx_count = ops.get('cx', 0) + ops.get('ecr', 0) + ops.get('cz', 0)

    print(f"  Transpiled depth: {transpiled.depth()}, gates: {gates} (2-qubit: {cx_count})")

    # Sample the circuit
    if use_hardware:
        sampler = SamplerV2(mode=active_backend)
        job = sampler.run([transpiled], shots=shots)
        result = job.result()
        creg_name = list(result[0].data.keys())[0]
        bitarray = getattr(result[0].data, creg_name)
        counts = bitarray.get_counts()
    else:
        sim = AerSimulator()
        job = sim.run(transpiled, shots=shots)
        counts = job.result().get_counts()

    total = sum(counts.values())
    probs = {bs: c / total for bs, c in counts.items()}

    info = {
        "qubits":   n_qubits,
        "gates":    gates,
        "cx_count": cx_count,
        "depth":    transpiled.depth(),
        "backend":  backend_label,
        "time_s":   elapsed,
    }
    return probs, opt, info

## Extract Groups from QAOA Output

Select the highest-probability bitstring with exactly $G-1$ ones. If QAOA finds no valid solution, fall back to the greedy choice: cut the $G-1$ cheapest gaps (lowest $w_i$).

In [46]:
def probs_to_groups(probs, sorted_custs, G, weights):
    n = len(weights)
    valid = {bs: p for bs, p in probs.items() if bs.count('1') == G-1}

    if valid:
        best_bs = max(valid, key=valid.get)
    else:
        cut_idxs = sorted(range(n), key=lambda i: weights[i])[:G-1]
        arr = list('0'*n)
        for i in cut_idxs:
            arr[n-1-i] = '1'
        best_bs = ''.join(arr)

    cuts = [i for i in range(n) if best_bs[n-1-i] == '1']

    groups, prev = [], 0
    for cut in sorted(cuts):
        groups.append(sorted_custs[prev:cut+1])
        prev = cut+1
    groups.append(sorted_custs[prev:])

    return [g for g in groups if g]

## Route Construction

Exact optimal ordering within each group via exhaustive permutation (feasible since group size $\leq C \leq 5$).

In [47]:
def build_route(group, dist_matrix):
    best, best_d = None, float('inf')
    for perm in permutations(group):
        route = [0] + list(perm) + [0]
        d = sum(dist_matrix[(route[k], route[k+1])] for k in range(len(route)-1))
        if d < best_d:
            best_d, best = d, route
    return best

def total_distance(routes, dist_matrix):
    return sum(dist_matrix[(r[k], r[k+1])] for _, r in routes for k in range(len(r)-1))


## Full Hybrid Solver

In [ ]:
def solve_cvrp_hybrid(depot, customers, N, C, reps=2, verbose=True):
    locations   = encode_locations(depot, customers)
    dist_matrix = build_distance_matrix(locations)
    H = len(customers)
    G = compute_num_groups(H, C, N)
    cust_ids     = [i for i in locations if i != 0]
    sorted_custs = angular_sort(cust_ids, locations)

    if verbose:
        print(f"  H={H} customers, G={G} group(s), C={C} capacity")

    if G == 1:
        groups = [sorted_custs]
        resource_info = {"qubits": 0, "gates": 0, "cx_count": 0, "depth": 0,
                         "backend": "none (G=1)", "time_s": 0.0, "valid_fraction": 1.0}
    else:
        weights  = build_cut_weights(sorted_custs, dist_matrix)
        cost_op  = build_cost_hamiltonian(weights, G)

        if verbose:
            print(f"  QAOA: {len(weights)} qubits, {G-1} cut(s) needed, reps={reps}")

        probs, opt, resource_info = run_qaoa(cost_op, reps=reps)

        # Fraction of shots that returned a valid bitstring (exactly G-1 ones)
        resource_info['valid_fraction'] = sum(p for bs, p in probs.items()
                                              if bs.count('1') == G - 1)
        groups = probs_to_groups(probs, sorted_custs, G, weights)

        if verbose:
            print(f"  QAOA: converged={opt.success}, energy={opt.fun:.4f}")
            print(f"  Valid bitstring fraction: {resource_info['valid_fraction']:.3f}")

    routes = [(i+1, build_route(g, dist_matrix)) for i, g in enumerate(groups)]
    dist   = total_distance(routes, dist_matrix)

    if verbose:
        for i, (_, route) in enumerate(routes):
            print(f"  r{i+1}: {' -> '.join(str(n) for n in route)}")
        print(f"  Total distance: {dist:.4f}")

    return routes, dist, resource_info

## Run All Instances

In [49]:
depot = (0, 0)

instances = {
    1: {"customers": [(-2,2),(-5,8),(2,3)],                                                                                "N": 2, "C": 5},
    2: {"customers": [(-2,2),(-5,8),(2,3)],                                                                                "N": 2, "C": 2},
    3: {"customers": [(-2,2),(-5,8),(2,3),(5,7),(2,4),(2,-3)],                                                             "N": 3, "C": 2},
    4: {"customers": [(-2,2),(-5,8),(6,3),(4,4),(3,2),(0,2),(-2,3),(-4,3),(2,3),(2,7),(-2,5),(-1,4)], "N": 4, "C": 3},
}

results = {}
resource_table = {}

for inst_num, params in instances.items():
    print(f"{'='*50}")
    print(f"Instance {inst_num}  |  N={params['N']}, C={params['C']}")
    routes, dist, res_info = solve_cvrp_hybrid(depot, params["customers"], params["N"], params["C"])
    results[inst_num] = {"routes": routes, "total_distance": dist}
    resource_table[inst_num] = res_info
    print()

Instance 1  |  N=2, C=5
  H=3 customers, G=1 group(s), C=5 capacity
  r1: 0 -> 3 -> 2 -> 1 -> 0
  Total distance: 21.7445

Instance 2  |  N=2, C=2
  H=3 customers, G=2 group(s), C=2 capacity
  QAOA: 2 qubits, 1 cut(s) needed, reps=2
  Running on: ibm_fez
    iteration 50
    iteration 100
    iteration 150
    iteration 200
  Optimization: 200 iterations in 0.1s
  Transpiled depth: 16, gate count: 31
  QAOA: converged=False, energy=-11.0959
  r1: 0 -> 3 -> 0
  r2: 0 -> 2 -> 1 -> 0
  Total distance: 26.1817

Instance 3  |  N=3, C=2
  H=6 customers, G=3 group(s), C=2 capacity
  QAOA: 5 qubits, 2 cut(s) needed, reps=2
  Running on: ibm_fez
  Optimization: 44 iterations in 0.1s
  Transpiled depth: 187, gate count: 322
  QAOA: converged=True, energy=-16.2160
  r1: 0 -> 6 -> 0
  r2: 0 -> 4 -> 0
  r3: 0 -> 3 -> 5 -> 2 -> 1 -> 0
  Total distance: 46.6202

Instance 4  |  N=4, C=3
  H=12 customers, G=4 group(s), C=3 capacity
  QAOA: 11 qubits, 3 cut(s) needed, reps=2
  Running on: aer_simulator 

## Resource Usage Table

In [50]:
print(f"{'Instance':<12} {'Qubits':<10} {'Gates':<10} {'Depth':<10} {'Backend':<20} {'Time (s)':<12}")
print('-' * 74)
for inst_num, info in resource_table.items():
    print(f"{inst_num:<12} {info['qubits']:<10} {info['gates']:<10} {info['depth']:<10} {info.get('backend','n/a'):<20} {info['time_s']:<12.2f}")


Instance     Qubits     Gates      Depth      Backend              Time (s)    
--------------------------------------------------------------------------
1            0          0          0          none                 0.00        
2            2          31         16         ibm_fez              0.14        
3            5          322        187        ibm_fez              0.07        
4            11         375        94         aer_simulator (local) 0.34        


## Noise Analysis

Three plots showing how hardware noise scales with problem size:

1. **Circuit resources** — transpiled depth and total gate count per instance. Both grow with the number of qubits due to the O(H²) ZZ-term structure of the cost Hamiltonian.
2. **Estimated circuit fidelity** — modelled as $F = (1-p_{2q})^{n_{2q}} \cdot (1-p_{1q})^{n_{1q}}$, where $p_{2q}$ and $p_{1q}$ are the average 2-qubit and 1-qubit gate error rates fetched directly from the ibm_fez backend properties. Fidelity decays exponentially with gate count.
3. **Valid bitstring fraction** — the fraction of the 2048 hardware shots that returned a bitstring with exactly $G-1$ ones (the constraint our QUBO enforces). A drop here indicates that noise is randomising measurements away from the feasible subspace, which directly degrades solution quality.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# --- Fetch ibm_fez gate error rates ---
try:
    props = hw_backend.properties()
    two_q_errs = [p.value for g in props.gates if len(g.qubits) == 2
                  for p in g.parameters if p.name == 'gate_error']
    one_q_errs = [p.value for g in props.gates if len(g.qubits) == 1
                  for p in g.parameters if p.name == 'gate_error']
    avg_2q = float(np.mean(two_q_errs)) if two_q_errs else 0.005
    avg_1q = float(np.mean(one_q_errs)) if one_q_errs else 0.0003
    print(f"{hw_backend.name} — avg 2q error: {avg_2q:.5f}, avg 1q error: {avg_1q:.5f}")
except Exception as e:
    avg_2q, avg_1q = 0.005, 0.0003
    print(f"Using default error rates (2q={avg_2q}, 1q={avg_1q}): {e}")

# --- Collect per-instance stats (skip Instance 1: QAOA not invoked) ---
hw_insts, depths, gates_list, cx_list, valid_fracs, fidelities = [], [], [], [], [], []

for inst, info in sorted(resource_table.items()):
    if info['qubits'] == 0 or info.get('depth') is None:
        continue
    cx  = info.get('cx_count', 0)
    sq  = info['gates'] - cx
    hw_insts.append(inst)
    depths.append(info['depth'])
    gates_list.append(info['gates'])
    cx_list.append(cx)
    fidelities.append((1 - avg_2q)**cx * (1 - avg_1q)**sq)
    vf = info.get('valid_fraction')
    valid_fracs.append(vf if vf is not None else float('nan'))

labels = [f'Instance {i}' for i in hw_insts]
x = np.arange(len(hw_insts))
w = 0.38

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Noise Analysis — QAOA on IBM Quantum Hardware', fontsize=13, fontweight='bold')

# Plot 1: Transpiled depth and total gate count
axes[0].bar(x - w/2, depths,      w, label='Circuit depth', color='steelblue',   alpha=0.85)
axes[0].bar(x + w/2, gates_list,  w, label='Total gates',   color='lightsalmon', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylabel('Count')
axes[0].set_title('Circuit Resources per Instance')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)
for xi, (d, g) in enumerate(zip(depths, gates_list)):
    axes[0].text(xi - w/2, d + 2,  str(d), ha='center', va='bottom', fontsize=8)
    axes[0].text(xi + w/2, g + 2,  str(g), ha='center', va='bottom', fontsize=8)

# Plot 2: Estimated circuit fidelity  F = (1-p_2q)^n_cx * (1-p_1q)^n_1q
axes[1].bar(x, fidelities, color='mediumseagreen', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel('Estimated fidelity')
axes[1].set_ylim(0, 1.1)
axes[1].set_title(f'Circuit Fidelity Estimate\n'
                  f'F = (1−p₂q)^n₂q · (1−p₁q)^n₁q\n'
                  f'p₂q={avg_2q:.4f}, p₁q={avg_1q:.4f}')
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.6, label='F=0.5 threshold')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)
for xi, f in enumerate(fidelities):
    axes[1].text(xi, f + 0.02, f'{f:.4f}', ha='center', va='bottom', fontsize=9)

# Plot 3: Valid bitstring fraction from hardware sampling
vf_plot = np.nan_to_num(valid_fracs)
axes[2].bar(x, vf_plot, color='mediumpurple', alpha=0.85)
axes[2].set_xticks(x); axes[2].set_xticklabels(labels)
axes[2].set_ylabel('Fraction of shots')
axes[2].set_ylim(0, 1.1)
axes[2].set_title('Valid Bitstring Fraction\n(exactly G−1 ones in measurement output)')
axes[2].grid(axis='y', alpha=0.3)
for xi, vf in enumerate(valid_fracs):
    label = f'{vf:.3f}' if not np.isnan(vf) else 'N/A'
    axes[2].text(xi, vf_plot[xi] + 0.02, label, ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('noise_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved noise_analysis.png")

## Write Solution Files

In [51]:
os.makedirs("solutions", exist_ok=True)

for inst_num, res in results.items():
    path = f"solutions/Instance{inst_num}.txt"
    with open(path, "w") as f:
        for i, (_, route) in enumerate(res["routes"], start=1):
            f.write(f"r{i}: " + ", ".join(str(n) for n in route) + "\n")
    print(f"Written {path}")

Written solutions/Instance1.txt
Written solutions/Instance2.txt
Written solutions/Instance3.txt
Written solutions/Instance4.txt
